# Phase 2 - DarkIR-lite Training

Fine-tunes DarkIR-m ("DarkIR-lite") on synthetic clean/dark EndoSLAM pairs.
`src/darkir_lite/model.py` + `train.py` were written against facts confirmed
in `notebooks/phase2a_explore` and validated locally against the real
cloned DarkIR repo + real checkpoint + synthetic frames (see PROGRESS.md).

**This run is a short smoke test** (`--max-steps 20`) to confirm the full
pipeline runs end-to-end on Kaggle's actual GPU/dataset before committing
to the full 20-epoch fine-tune -- keeps GPU-quota usage minimal for this
first run. **GPU is on** this time (`enable_gpu: true`), unlike every
prior notebook in this project.

## 0. Setup: clone our repo + DarkIR, install deps

In [ ]:
REPO_URL = "https://github.com/ritiksharma3/endoslam.git"
DARKIR_URL = "https://github.com/cidautai/DarkIR.git"

!git clone $REPO_URL repo
!git clone $DARKIR_URL repo/DarkIR_upstream

%cd repo
!pip install -q -r environment/requirements.txt

## 1. Resolve dataset mount + GPU check

In [ ]:
import os
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

def find_endoslam_root(base="/kaggle/input", max_depth=4):
    for root, dirs, _files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        if os.path.basename(root).lower() == "endoslam":
            return root
    return None

DATA_ROOT = find_endoslam_root()
assert DATA_ROOT, "could not find an endoslam dir under /kaggle/input"
print("DATA_ROOT:", DATA_ROOT)

## 2. Write a run-specific config (data.root filled in) and smoke-test train.py

In [ ]:
import yaml

with open("configs/config.yaml") as f:
    config = yaml.safe_load(f)
config["data"]["root"] = DATA_ROOT

RUN_CONFIG_PATH = "/kaggle/working/run_config.yaml"
with open(RUN_CONFIG_PATH, "w") as f:
    yaml.safe_dump(config, f)
print(f"wrote {RUN_CONFIG_PATH} with data.root = {DATA_ROOT}")

In [ ]:
!python -m src.darkir_lite.train \
    --config /kaggle/working/run_config.yaml \
    --output-dir /kaggle/working/checkpoints \
    --max-steps 20

## 3. Confirm checkpoints were written

In [ ]:
import os

ckpt_dir = "/kaggle/working/checkpoints"
files = sorted(os.listdir(ckpt_dir)) if os.path.isdir(ckpt_dir) else []
print("checkpoint files:", files)
assert files, "expected at least one checkpoint file after the smoke test"

import torch
latest = os.path.join(ckpt_dir, sorted(files)[-1])
ckpt = torch.load(latest, map_location="cpu", weights_only=False)
print(f"loaded {latest}: epoch={ckpt['epoch']}, global_step={ckpt['global_step']}, "
      f"val_psnr={ckpt.get('val_psnr')}, val_ssim={ckpt.get('val_ssim')}")

## Done

If this completed without error and produced checkpoints with sensible
(non-NaN) PSNR/SSIM, the pipeline is smoke-tested. Record results in
`PROGRESS.md`, then remove `--max-steps 20` (or raise it) for the real
20-epoch fine-tune -- that full run is out of scope for this notebook's
first pass.